In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_comparison/train_max.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_comparison/val_fixed.csv'

# ★ ここを変更することでモデルを切り替えられます ('CustomCNN' または 'ConvNeXt')
# MODEL_TYPE = 'ConvNeXt' 
MODEL_TYPE = 'CustomCNN' 

TARGET_EPOCH = 100
SAVE_DIR = f"../save/{MODEL_TYPE}_Only_{TARGET_EPOCH}epoch_fixed_dataset"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_model.pth'


# --- 2. モデルの定義 ---
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlock, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        return torch.relu(self.conv_block(x) + self.shortcut(x))

class CustomCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResBlock(32, 64),
            ResBlock(64, 64)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

class ConvNeXtOnly(nn.Module):
    def __init__(self, num_classes=3):
        super(ConvNeXtOnly, self).__init__()
        self.model = models.convnext_tiny(weights=None)
        self.model.classifier[2] = nn.Linear(self.model.classifier[2].in_features, num_classes)
        
    def forward(self, x):
        return self.model(x)


# --- 3. Datasetの定義 ---
class ImageOnlyDataset(Dataset):
    def __init__(self, csv_path, img_root, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.transform = transform
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label


# --- 4. ユーティリティ: 学習曲線保存 ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


# --- 5. メイン学習関数 ---
def train_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    print(f"=== Starting Training with {MODEL_TYPE} ===")
    print(f"Save Directory: {SAVE_DIR}")

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = ImageOnlyDataset(TRAIN_CSV_PATH, IMG_ROOT, transform=transform)
    val_dataset   = ImageOnlyDataset(VAL_CSV_PATH, IMG_ROOT, transform=transform)
    
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False)

    if MODEL_TYPE == 'CustomCNN':
        model = CustomCNN(num_classes=3).to(DEVICE)
    elif MODEL_TYPE == 'ConvNeXt':
        model = ConvNeXtOnly(num_classes=3).to(DEVICE)
    else:
        raise ValueError("MODEL_TYPE must be 'CustomCNN' or 'ConvNeXt'")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    print(f"\n--- Training Phase ({TARGET_EPOCH} epochs) ---")
    history = {'train_loss': [], 'val_acc': []}
    best_acc = 0.0

    for epoch in range(TARGET_EPOCH):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_dataset)
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data).item()
                
        val_acc = correct / total
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{TARGET_EPOCH} | Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_WEIGHT_PATH)
            print(f"  --> Best Model Saved (Acc: {best_acc:.4f})")

    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    print("Training Complete.\n")

if __name__ == "__main__":
    train_model()

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_comparison/test_fixed.csv'

# ★ 学習時と同じモデルタイプを指定してください
MODEL_TYPE = 'ConvNeXt' 
MODEL_TYPE = 'CustomCNN' 

TARGET_EPOCH = 100
SAVE_DIR = f"../save/{MODEL_TYPE}_Only_{TARGET_EPOCH}epoch_fixed_dataset"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_model.pth'


# --- 2. モデルの定義 (ロード用に必要) ---
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlock, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        return torch.relu(self.conv_block(x) + self.shortcut(x))

class CustomCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResBlock(32, 64),
            ResBlock(64, 64)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

class ConvNeXtOnly(nn.Module):
    def __init__(self, num_classes=3):
        super(ConvNeXtOnly, self).__init__()
        self.model = models.convnext_tiny(weights=None)
        self.model.classifier[2] = nn.Linear(self.model.classifier[2].in_features, num_classes)
        
    def forward(self, x):
        return self.model(x)


# --- 3. Datasetの定義 ---
class ImageOnlyDataset(Dataset):
    def __init__(self, csv_path, img_root, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.transform = transform
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label


# --- 4. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}\n先に学習を完了させてください。")
        return

    print(f"=== Starting Test Evaluation for {MODEL_TYPE} ===")

    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_dataset = ImageOnlyDataset(TEST_CSV_PATH, IMG_ROOT, transform=transform)
    test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False)

    if MODEL_TYPE == 'CustomCNN':
        model = CustomCNN(num_classes=3).to(DEVICE)
    elif MODEL_TYPE == 'ConvNeXt':
        model = ConvNeXtOnly(num_classes=3).to(DEVICE)

    # 学習済み重みのロード
    model.load_state_dict(torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\nOverall Test Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 混同行列の保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix ({MODEL_TYPE} Only)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    cm_save_path = f'{SAVE_DIR}/test_confusion_matrix.png'
    plt.savefig(cm_save_path)
    plt.close()
    
    print(f"Confusion matrix saved to {cm_save_path}")
    print("Test Evaluation Finished.")

if __name__ == "__main__":
    evaluate_model()

=== Starting Test Evaluation for ConvNeXt ===


FileNotFoundError: [Errno 2] No such file or directory: '/home/hiromu/energy/src/FF/AFF/dataset_comparison/test_fixed.csv'

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

# --- 設定: ファイルパス ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
VAL_CSV_PATH   = f'{pre}/src/FF/AFF/dataset_upsamp/val_fixed.csv'

# 学習設定
TARGET_EPOCH = 100
# 保存先ディレクトリ名をベースライン用に変更
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_baseline_image_only_scratch"

# ★ フラグ設定
USE_SCHEDULER = False # Trueでコサイン波状に学習率を落とす、Falseで固定

# --- 1. Datasetクラス (画像のみ版) ---
class ImageOnlyDataset(Dataset):
    def __init__(self, dataframe, img_root, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.labels = self.df['Label'].values

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # CSVのファイル名にある '_mask' を '_combined' に置換
        original_filename = row['filename']
        actual_filename = original_filename.replace('_mask', '_combined')
        
        # 'combined' フォルダを参照
        img_path = os.path.join(self.img_root, 'combined', row['category'], actual_filename)
        
        try: 
            image = Image.open(img_path).convert('RGB')
        except Exception as e: 
            # print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        # 数値データは返さず、画像とラベルのみ返す
        return image, label

# --- 2. ユーティリティ ---
def save_learning_curve(history, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'g-', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'b-', label='Val Acc')
    plt.title('Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# --- 3. メイン学習関数 ---
def train_model():
    BATCH_SIZE = 16
    EPOCHS = TARGET_EPOCH
    LR = 0.0001
    WD = 0.05
    NUM_CLASSES = 3
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    best_model_path = f'{SAVE_DIR}/best_baseline_model.pth'

    # 訓練用 (データ拡張あり)
    train_transform = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_df, val_df = pd.read_csv(TRAIN_CSV_PATH), pd.read_csv(VAL_CSV_PATH)
    
    # Datasetインスタンス化 (Scalerなどは不要)
    train_ds = ImageOnlyDataset(train_df, IMG_ROOT, transform=train_transform)
    val_ds = ImageOnlyDataset(val_df, IMG_ROOT, transform=val_transform)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- 公式の純粋なConvNeXtをロード (★事前学習なし) ---
    print("Loading official ConvNeXt-Tiny (No Pre-training)...")
    model = models.convnext_tiny(weights=None)
    
    # 最終層 (Classifier) を3クラス用に変更
    # ConvNeXtの classifier[2] が最後のLinear層になっています (入力次元は768)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
    
    model = model.to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    
    if USE_SCHEDULER:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        print("Scheduler: ENABLED (Cosine Annealing)")
    else:
        scheduler = None
        print("Scheduler: DISABLED (Fixed Learning Rate)")
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0

    print(f"Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        running_loss = 0.0
        # hc_feats が無くなったので unpack する変数を2つに変更
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        if scheduler is not None:
            scheduler.step()
            
        epoch_train_loss = running_loss / len(train_ds)

        # --- Val ---
        model.eval()
        v_running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                
                outputs = model(images)
                v_loss = criterion(outputs, labels)
                v_running_loss += v_loss.item() * images.size(0)
                
                _, preds = torch.max(outputs, 1)
                total += labels.size(0)
                correct += torch.sum(preds == labels.data)
        
        epoch_val_loss = v_running_loss / len(val_ds)
        val_acc = (correct.double() / total).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(val_acc)
        
        curr_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {curr_lr:.6f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Best Model Saved! (Acc: {best_acc:.4f})")
            
        if (epoch + 1) % 5 == 0: 
            save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')

    print(f"Training Complete. Best Val Acc: {best_acc:.4f}")
    save_learning_curve(history, f'{SAVE_DIR}/learning_curve.png')
    return history

if __name__ == "__main__":
    train_model()

Loading official ConvNeXt-Tiny (No Pre-training)...
Scheduler: DISABLED (Fixed Learning Rate)
Starting Training on cuda...
Epoch 1/100 | Train Loss: 1.0964 | Val Loss: 0.9724 | Val Acc: 0.4306 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.4306)
Epoch 2/100 | Train Loss: 0.9159 | Val Loss: 0.8458 | Val Acc: 0.6250 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.6250)
Epoch 3/100 | Train Loss: 0.8382 | Val Loss: 0.7006 | Val Acc: 0.7014 | LR: 0.000100
  --> Best Model Saved! (Acc: 0.7014)
Epoch 4/100 | Train Loss: 0.8521 | Val Loss: 0.7587 | Val Acc: 0.6181 | LR: 0.000100
Epoch 5/100 | Train Loss: 0.7927 | Val Loss: 0.6746 | Val Acc: 0.6944 | LR: 0.000100
Epoch 6/100 | Train Loss: 0.7528 | Val Loss: 0.6688 | Val Acc: 0.6736 | LR: 0.000100
Epoch 7/100 | Train Loss: 0.7446 | Val Loss: 0.6682 | Val Acc: 0.6806 | LR: 0.000100
Epoch 8/100 | Train Loss: 0.7622 | Val Loss: 0.6908 | Val Acc: 0.6875 | LR: 0.000100
Epoch 9/100 | Train Loss: 0.7064 | Val Loss: 0.8932 | Val Acc: 0.5556 | LR: 0.00010

In [6]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp_clean/test_fixed.csv' # ※テスト用CSVのパスを指定

TARGET_EPOCH = 100
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_baseline_image_only_scratch"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_baseline_model.pth'

# --- 3. Datasetの定義 (画像のみ版) ---
class ImageOnlyDataset(Dataset):
    def __init__(self, csv_path, img_root, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.transform = transform
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        original_filename = row['filename']
        actual_filename = original_filename.replace('_mask', '_combined')
        img_path = os.path.join(self.img_root, 'combined', row['category'], actual_filename)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError) as e:
            print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0)) # 学習時と同じ224x224
            
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

# --- 4. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}\n先に学習を完了させてください。")
        return

    print("=== Starting Test Evaluation for ConvNeXt Scratch Baseline ===")

    # テストデータ用の変換 (リサイズと正規化のみ、拡張はなし)
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_dataset = ImageOnlyDataset(TEST_CSV_PATH, IMG_ROOT, transform=test_transform)
    test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

    # モデルの準備と重みのロード
    # モデルの準備と重みのロード
    model = models.convnext_tiny(weights=None)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, 3) # 3クラス分類
    model = model.to(DEVICE)
    
    model.load_state_dict(torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE))
    model.eval()

    all_preds, all_labels = [], []
    
    print("Running inference on test dataset...")
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 結果の算出と表示
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\nOverall Test Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 混同行列の保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (ConvNeXt Scratch Baseline)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    cm_save_path = f'{SAVE_DIR}/test_confusion_matrix.png'
    plt.savefig(cm_save_path)
    plt.close()
    
    print(f"Confusion matrix saved to {cm_save_path}")
    print("Test Evaluation Finished.")

if __name__ == "__main__":
    evaluate_model()

=== Starting Test Evaluation for ConvNeXt Scratch Baseline ===
Running inference on test dataset...

Overall Test Accuracy: 0.8662

Classification Report:
              precision    recall  f1-score   support

           0     0.9362    0.9565    0.9462        46
           1     0.7458    0.9167    0.8224        48
           2     0.9722    0.7292    0.8333        48

    accuracy                         0.8662       142
   macro avg     0.8847    0.8675    0.8673       142
weighted avg     0.8840    0.8662    0.8662       142

Confusion matrix saved to ../save/CBAMtest/ConvNext_100epoch_baseline_image_only_scratch/test_confusion_matrix.png
Test Evaluation Finished.
